<a id="top"></a>
# House Sales: preparazione dati e primo modello di regressione (esercitazione)

Nel modulo 1 hai usato il dataset House Sales per inquadrare il problema: stimare il **prezzo di vendita** delle case in una certa area.  
Nel modulo 2, con il caso Bank Marketing, hai lavorato sulla preparazione di un dataset per un problema di **classificazione binaria**.

Qui resti nel contesto naturale di House Sales: **regressione sul prezzo**.  
L’obiettivo è replicare in versione compatta la pipeline di preparazione dati vista nel marketing bancario e collegarla a un primo modello lineare.

<a id="obiettivo-esercitazione"></a>
## Obiettivo dell'esercitazione

Nel caso Bank Marketing ti sei concentrato su una **classificazione**: prevedere se un cliente sottoscrive o meno un deposito (output binario 0/1).  
Nel caso House Sales torni al problema di **regressione**: il modello prevede un **numero continuo** (il prezzo), non una classe.

Riassumendo in due righe la differenza:

- **Classificazione**: l’output è una categoria (es. _sì/no_, _classe A/B/C_).  
- **Regressione**: l’output è un valore continuo (es. prezzo in euro).

In questa esercitazione:

- preparerai le feature in modo coerente con quanto fatto sul marketing (imputazione + normalizzazione robusta);
- allenerai un modello lineare (`LinearRegression`) per prevedere il prezzo;



<a id="caricamento-dataset"></a>
## Caricamento del dataset House Sales

Userai lo stesso dataset del notebook House Sales del modulo 1.  
Il dataset è su OpenML (id 42092) e si può scaricare con `fetch_openml` di scikit-learn.

- `fetch_openml` scarica un dataset da OpenML e restituisce un oggetto da cui puoi ricavare un DataFrame.  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html

Nel blocco seguente carichi i dati in un DataFrame `house` e dai un’occhiata alle prime righe.


In [9]:
# Importa pandas per gestire i dati in forma tabellare (DataFrame).
import pandas as pd
# Importa la funzione di scikit-learn che scarica dataset da OpenML.
from sklearn.datasets import fetch_openml

# Scarica House Sales (ID OpenML 42092); as_frame=True richiede dati in formato pandas.
raw = fetch_openml(data_id=42092, as_frame=True)
# Copia la tabella che contiene le feature e la colonna prezzo.
house = raw.frame.copy()

# Controlla quante osservazioni (righe) e variabili (colonne) sono state caricate.
print("Dimensioni del dataset (righe, colonne):", house.shape)
# Mostra le prime righe per verificare la struttura e alcuni valori del dataset.
house.head()


Dimensioni del dataset (righe, colonne): (21613, 20)


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


# Step 1: Preparazione del dato

Nel primo step dovrai preparare il dato:
1. separare X e y 
2. rimuovere le features inutilizzabili (date, zipcode, lat, long, yr_built, yr_renovated)
3. train test split (30% di test)


<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Attenzione**

In questo contesto ci accontentiamo di usare le variabili così come sono per faciilitare le prossime fasi. Ma la pratica del *feature engineering* prevede l'elaborazione di nuove variabili più utiili:
- yr_built -> età 
- yr_nenovated -> renovated?
- sqft_above, sqft_basement-> metri quadri totali  

</div>

In [24]:
# Importa la configurazione che permette ai trasformatori di restituire DataFrame.
from sklearn import set_config
# Importa la funzione che divide i dati in insiemi di training e test.
from sklearn.model_selection import train_test_split
# Mantiene i nomi delle colonne durante il preprocessing (output in formato pandas).
set_config(transform_output="pandas")


In [16]:
# Elenca le colonne che l'esercitazione esclude dalle feature usate dal modello.
colonne_da_rimuovere = [
    "date", "zipcode", "lat", "long", "yr_built", "yr_renovated"
]

# Crea X con le variabili esplicative, rimuovendo sia il target sia le colonne escluse.
X = house.drop(columns=["price", *colonne_da_rimuovere])
# Crea y con il valore che il modello dovrà imparare a prevedere: il prezzo.
y = house["price"]

# Divide feature e target: 70% per addestrare, 30% per valutare su dati separati.
# random_state rende sempre uguale la divisione quando si riesegue il notebook.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# Stampa le dimensioni per verificare che X e y siano stati divisi coerentemente.
print("Dimensioni training:", X_train.shape, y_train.shape)
print("Dimensioni test:", X_test.shape, y_test.shape)

Dimensioni training: (15129, 13) (15129,)
Dimensioni test: (6484, 13) (6484,)


## Step 2 : Pipeline di preprocessing 

Dovrai creare una pipeline di scikit composta da `SimpleImputer` e `RobustScaler`. In questo caso **non** ti servirà usare `ColumnTrasformer` perché tutte le features sono numeriche (verifica). 

Usa la mediana per l'imputazione di tutte le features:
- ("imputer", SimpleImputer(strategy="median"))
- https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html

In [19]:
# Importa Pipeline per concatenare i passaggi di preprocessing in sequenza.
from sklearn.pipeline import Pipeline
# Importa lo strumento che sostituisce eventuali valori mancanti.
from sklearn.impute import SimpleImputer
# Importa lo scaler robusto, meno sensibile ai valori estremi.
from sklearn.preprocessing import RobustScaler


In [20]:
# Seleziona i nomi delle colonne di X_train che non hanno un tipo numerico.
colonne_non_numeriche = X_train.select_dtypes(exclude="number").columns
# Un elenco vuoto conferma che tutte le feature possono essere trattate numericamente.
print("Colonne non numeriche:", list(colonne_non_numeriche))

# Definisce il preprocessing in ordine: prima imputazione, poi normalizzazione robusta.
preprocessing_pipeline = Pipeline(steps=[
    # Sostituisce eventuali valori mancanti di ogni feature con la sua mediana.
    ("imputer", SimpleImputer(strategy="median")),
    # Centra e scala le feature usando statistiche robuste agli outlier.
    ("scaler", RobustScaler())
])

Colonne non numeriche: []


In [25]:
# Stampa un riepilogo di tutto ciò che è stato fatto fino a questo punto
print("=== RIASSUNTO PIPELINE HOUSE SALES ===")
print(f"Dataset caricato: house.shape = {house.shape}")
print(f"Feature X: {X.shape}, target y: {y.shape}")
print(f"Train/Test split: X_train={X_train.shape}, X_test={X_test.shape}, y_train={y_train.shape}, y_test={y_test.shape}")
print(f"Colonne rimosse: {colonne_da_rimuovere}")
print(f"Pipeline preprocessing: {preprocessing_pipeline.named_steps}")
print(f"Pipeline completa: {model_pipeline}")
print(f"ROC AUC sul test (soglia mediana del training): {roc_auc_test:.3f}")
print("\nAnteprima confronto reali vs previsti:")
print(confronto.head().to_string(index=True))

=== RIASSUNTO PIPELINE HOUSE SALES ===
Dataset caricato: house.shape = (21613, 20)
Feature X: (21613, 13), target y: (21613,)
Train/Test split: X_train=(15129, 13), X_test=(6484, 13), y_train=(15129,), y_test=(6484,)
Colonne rimosse: ['date', 'zipcode', 'lat', 'long', 'yr_built', 'yr_renovated']
Pipeline preprocessing: {'imputer': SimpleImputer(strategy='median'), 'scaler': RobustScaler()}
Pipeline completa: Pipeline(steps=[('preprocessing',
                 Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                 ('scaler', RobustScaler())])),
                ('regression', LinearRegression())])
ROC AUC sul test (soglia mediana del training): 0.847

Anteprima confronto reali vs previsti:
       Prezzo reale  Prezzo previsto
735        365000.0     5.539663e+05
2830       865000.0     7.523221e+05
4106      1038000.0     1.251868e+06
16218     1490000.0     1.519499e+06
19964      711000.0     7.280434e+05


## Step 3: Aggiungi anche `LinearRegression` come estimator in fondo alla pipeline. Produci delle previsioni sul test

In [ ]:
# Importa Pipeline per costruire la pipeline completa del modello.
from sklearn.pipeline import Pipeline
# Importa il modello lineare di regressione che predice un prezzo continuo.
from sklearn.linear_model import LinearRegression


In [22]:
# Crea una pipeline completa: trasforma le feature e poi applica la regressione.
model_pipeline = Pipeline(steps=[
    # Esegue imputazione e scaling definiti nella pipeline di preprocessing.
    ("preprocessing", preprocessing_pipeline),
    # Impara una relazione lineare tra le feature e il prezzo.
    ("regression", LinearRegression())
])

# Adatta tutti i passaggi ai dati di training e apprende i coefficienti del modello.
model_pipeline.fit(X_train, y_train)
# Usa il modello addestrato per stimare il prezzo delle case nell'insieme di test.
y_pred = model_pipeline.predict(X_test)

# Affianca prezzi osservati e previsti; gli indici consentono di associare le righe.
confronto = pd.DataFrame({
    "Prezzo reale": y_test,
    "Prezzo previsto": y_pred
})
# Visualizza le prime righe del confronto.
confronto.head()

,Prezzo reale,Prezzo previsto
735,365000.0,5.539663e+05
2830,865000.0,7.523221e+05
4106,1038000.0,1.251868e+06
16218,1490000.0,1.519499e+06
19964,711000.0,7.280434e+05


In [23]:
# Importa la metrica AUC, qui usata per misurare la capacità di ordinare le case costose.
from sklearn.metrics import roc_auc_score

# Calcola la soglia usando solo il training set, evitando di ricavarla dal test.
soglia_prezzo = y_train.median()
# Converte i prezzi reali del test in etichette: 1 sopra la mediana, altrimenti 0.
y_test_costosa = (y_test > soglia_prezzo).astype(int)

# Usa i prezzi previsti come punteggi continui e valuta quanto bene separano le due classi.
roc_auc_test = roc_auc_score(y_test_costosa, y_pred)
# Stampa l'AUC con tre cifre decimali.
print(f"ROC AUC sul test (prezzo sopra la mediana): {roc_auc_test:.3f}")

ROC AUC sul test (prezzo sopra la mediana): 0.847


In [27]:
from sklearn.metrics import classification_report

# Converte i prezzi previsti in classi usando la soglia calcolata sul training.
y_pred_costosa = (y_pred > soglia_prezzo).astype(int)

# La ROC AUC usa i punteggi continui, non le classi previste.
roc_auc_test = roc_auc_score(y_test_costosa, y_pred)
print(f"ROC AUC (test): {roc_auc_test:.3f}\n")

print(classification_report(
    y_test_costosa,
    y_pred_costosa,
    target_names=["0", "1"]
))

ROC AUC (test): 0.847

              precision    recall  f1-score   support

           0       0.79      0.71      0.75      3246
           1       0.73      0.81      0.77      3238

    accuracy                           0.76      6484
   macro avg       0.76      0.76      0.76      6484
weighted avg       0.76      0.76      0.76      6484



In [ ]:
## Riepilogo: cosa è stato fatto

1. **Caricamento dei dati**  
    Il dataset House Sales è stato scaricato da OpenML e copiato nel DataFrame `house`.

2. **Separazione di feature e target**  
    La colonna `price`, cioè il prezzo da prevedere, è stata assegnata a `y`. Le altre colonne sono state assegnate a `X`, escludendo `date`, `zipcode`, `lat`, `long`, `yr_built` e `yr_renovated`.

3. **Divisione in training e test**  
    I dati sono stati suddivisi in un insieme di training (70%) e uno di test (30%), usando `random_state=42` per rendere la divisione riproducibile.

4. **Preprocessing delle feature**  
    È stata creata una pipeline che sostituisce eventuali valori mancanti con la mediana (`SimpleImputer`) e applica una normalizzazione robusta (`RobustScaler`). Entrambi i passaggi vengono appresi sui dati di training.

5. **Addestramento e previsione**  
    La pipeline completa applica il preprocessing e poi addestra una `LinearRegression`. Il modello è stato allenato su `X_train` e `y_train`, quindi usato per prevedere i prezzi di `X_test`.

6. **Confronto e valutazione**  
    Le previsioni sono state affiancate ai prezzi reali in `confronto`. Infine, i prezzi reali del test sono stati trasformati in due classi usando la mediana del training come soglia; la ROC AUC misura quanto bene le previsioni ordinano le case costose sopra quelle non costose. È una valutazione derivata in forma di classificazione, non una metrica standard di regressione.

## Extra: se hai concluso puoi provare ad esplorare:

1. cambia strategia di imputing (mean invece di median)
2. cambia modello, from sklearn.ensemble import GradientBoostingRegressor (info: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html#sklearn.ensemble.GradientBoostingRegressor)

Molto avanzato: 
1. Fai uno stacking dei due modelli che hai allenato (https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html)
